In [ ]:
#import sys
#sys.path.append('/kaggle/input/my-trees')
import pandas as pd

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from boosting import GradientBoostingClassifier as MyGradientBoostingClassifier
from trees import DecisionTreeClassifier as MyDecisionTreeClassifier
from forest import RandomForestClassifier as MyRandomForestClassifier

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline 
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


## Загрузка и предобработка данных

In [2]:
try:
    df = pd.read_csv('../datasets/training.csv')
except:
    df = pd.read_csv('/kaggle/input/ml5-data/training.csv')

In [3]:
df = df.drop(['RefId'], axis=1)
df['PRIMEUNIT'] = df['PRIMEUNIT'].fillna('UNKNOWN')
df['AUCGUART'] = df['AUCGUART'].fillna('UNKNOWN')

df = df.drop('WheelTypeID', axis=1)
df['WheelType'] = df['WheelType'].fillna('UNKNOWN')
df['Trim'] = df['Trim'].fillna('UNKNOWN')

df = df.dropna()

### Разделение на выборки 

In [4]:
df['PurchDate'] = pd.to_datetime(df['PurchDate'])
df = df.sort_values(by='PurchDate', ascending=True)
train_val, test = train_test_split(df, shuffle=False, test_size=1/3)
train, val = train_test_split(train_val, shuffle=False, test_size=0.5)
del train_val

In [5]:
train_end = '2009-09-15'
val_end = '2010-05-14'

train = df[df['PurchDate'] <= train_end]
val = df[(df['PurchDate'] > train_end) & (df['PurchDate'] <= val_end)]
test = df[df['PurchDate'] > val_end]

In [6]:
train = train.drop('PurchDate', axis=1)
val = val.drop('PurchDate', axis=1)
test = test.drop('PurchDate', axis=1)

### Кодирование фичей

In [7]:
def build_preprocess_pipeline(train_df: pd.DataFrame):
    cat_cols = [c for c, dt in zip(train_df.columns, train_df.dtypes) if dt.kind not in ['i', 'f'] and c != 'IsBadBuy']
    num_cols = [c for c, dt in zip(train_df.columns, train_df.dtypes) if dt.kind in ['i', 'f'] and c != 'IsBadBuy']

    preproc = ColumnTransformer(
        transformers=[
            ('num', Pipeline(steps=[('scaler', StandardScaler())]), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ],
        remainder='drop'
    )
    return preproc

In [8]:
def encode_features(preproc, df):
    X = pd.DataFrame(data=preproc.transform(df.drop('IsBadBuy', axis=1)),
                     columns=preproc.get_feature_names_out())
    y = df['IsBadBuy']
    return X, y

In [9]:
preproc = build_preprocess_pipeline(train)
preproc.fit(train)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 ['VehYear', 'VehicleAge', 'VehOdo',
                                  'MMRAcquisitionAuctionAveragePrice',
                                  'MMRAcquisitionAuctionCleanPrice',
                                  'MMRAcquisitionRetailAveragePrice',
                                  'MMRAcquisitonRetailCleanPrice',
                                  'MMRCurrentAuctionAveragePrice',
                                  'MMRCurrentAuctionCleanPrice',
                                  'MMRCurrentRetailAveragePrice',
                                  'MMRCurrentRetailCleanPrice', 'BYRNO',
                                  'VNZIP1', 'VehBCost', 'IsOnlineSale',
                                  'WarrantyCost']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Auction', 'Make', 'Model', 'Trim',
                                  'SubModel', 'Color', 'Transmission',
                                  'WheelType', 'Nationality', 'Size',
                                  'TopThreeAmericanName', 'PRIMEUNIT',
                                  'AUCGUART', 'VNST'])])

In [10]:
X_train, y_train = encode_features(preproc, train)
X_val, y_val = encode_features(preproc, val)
X_test, y_test = encode_features(preproc, test)

### Версии моделей

#### Деревья решений

In [11]:
def gini(actual, probs):
    auc = roc_auc_score(actual, probs)
    return 2 * auc - 1

In [12]:
my_tree = MyDecisionTreeClassifier()
my_tree.fit(X_train, y_train)

In [13]:
probs = my_tree.predict_proba(X_val)[:, 1]
gini(y_val, probs)

0.42512118133639576

In [14]:
sklearn_tree = DecisionTreeClassifier(max_depth=7)
sklearn_tree.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=7)

In [15]:
probs = sklearn_tree.predict_proba(X_val)[:, 1]
gini(y_val, probs)

0.43875230778160845

Версия из sklearn слегка лучше, вероятно, просто больше критериев остановки, поэтому дерево меньше переобучается.

### Случайный лес

In [16]:
my_random_forest = MyRandomForestClassifier(max_depth=7, random_state=123)
my_random_forest.fit(X_train, y_train)

probs = my_random_forest.predict_proba(X_val)[:, 1]
gini(y_val, probs)

0.4205599407621117

In [17]:
sklearn_random_forest = RandomForestClassifier(max_depth=7, random_state=123)
sklearn_random_forest.fit(X_train, y_train)

probs = sklearn_random_forest.predict_proba(X_val)[:, 1]
gini(y_val, probs)

0.45791409868833766

#### Градиентный бустинг

In [18]:
my_gradient_boosting = MyGradientBoostingClassifier(max_features='sqrt', random_state=123)
my_gradient_boosting.fit(X_train, y_train)

probs = my_gradient_boosting.predict_proba(X_val)[:, 1]
gini(y_val, probs)

0.4183451017997084

In [19]:
catboost_cl = CatBoostClassifier(
        iterations=10000,         
        learning_rate=0.05,
        depth=6,
        eval_metric='AUC',         
        early_stopping_rounds=50, 
        use_best_model=True,       
        verbose=100                 
)

catboost_cl.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),  
    plot=True                   
)

probs = catboost_cl.predict_proba(X_val)[:,1]
gini(y_val, probs)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	test: 0.6999451	best: 0.6999451 (0)	total: 69.6ms	remaining: 11m 35s
100:	test: 0.7423726	best: 0.7425821 (99)	total: 1.57s	remaining: 2m 33s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7452310066
bestIteration = 130

Shrink model to first 131 iterations.


0.4904620131798525

In [20]:
xgb_class = XGBClassifier(
    #DART=True,
    n_estimators=1000,  
    learning_rate=0.1,
    max_depth=6,
    eval_metric='auc',
    early_stopping_rounds=50, 
    verbose=True 
)

xgb_class.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
)

probs = xgb_class.predict_proba(X_val)[:,1]
gini(y_val, probs)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [00:23:19] WARNING: /workspace/src/learner.cc:742: 
Parameters: { "verbose" } are not used.

  warnings.warn(smsg, UserWarning)


[0]	validation_0-auc:0.72552
[1]	validation_0-auc:0.72412
[2]	validation_0-auc:0.73317
[3]	validation_0-auc:0.73587
[4]	validation_0-auc:0.73514
[5]	validation_0-auc:0.73583
[6]	validation_0-auc:0.73600
[7]	validation_0-auc:0.73551
[8]	validation_0-auc:0.73648
[9]	validation_0-auc:0.73757
[10]	validation_0-auc:0.73870
[11]	validation_0-auc:0.73905
[12]	validation_0-auc:0.73999
[13]	validation_0-auc:0.74110
[14]	validation_0-auc:0.74156
[15]	validation_0-auc:0.74123
[16]	validation_0-auc:0.74129
[17]	validation_0-auc:0.74146
[18]	validation_0-auc:0.74219
[19]	validation_0-auc:0.74244
[20]	validation_0-auc:0.74275
[21]	validation_0-auc:0.74302
[22]	validation_0-auc:0.74267
[23]	validation_0-auc:0.74255
[24]	validation_0-auc:0.74303
[25]	validation_0-auc:0.74305
[26]	validation_0-auc:0.74326
[27]	validation_0-auc:0.74442
[28]	validation_0-auc:0.74428
[29]	validation_0-auc:0.74422
[30]	validation_0-auc:0.74450
[31]	validation_0-auc:0.74463
[32]	validation_0-auc:0.74443
[33]	validation_0-au

0.49350151988737334

In [21]:
lgmb_class = LGBMClassifier(
    n_estimators=1000,         
    learning_rate=0.1,
    early_stopping_round=50,
    verbose=100,
    #linear_tree = True
)

lgmb_class.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
)


probs = lgmb_class.predict_proba(X_val)[:,1]
gini(y_val, probs)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 2766, number of negative: 21357
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.920760
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.134284
[LightGBM] [Debug] init for col-wise cost 0.001818 seconds, init for row-wise cost 0.002552 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002868 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Dense Multi-Val Bin
[LightGBM] [Info] Total Bins 3965
[LightGBM] [Info] Number of data points in the train set: 24123, number of used features: 534
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.114662 -> initscore=-2.043977
[LightGBM] [Info] Start training f

0.48243142012104134

### Лучшая модель 

In [22]:
probs = catboost_cl.predict_proba(X_test)[:,1]
gini(y_test, probs)

0.45315621844897236

Я бы не сказал, что это оверфит, скорее шифт в данных из-за временной специфики. 